# Package

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Optimize memory

In [2]:
def reduce_mem_usage(df):
    """ Ép kiểu dữ liệu để giảm RAM """
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory usage of dataframe is {start_mem:.2f} MB')
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object and col not in ['id']: # Bỏ qua cột ID dạng chuỗi
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float32) # Dùng float32 thay vì 16 để tránh lỗi số học
                else:
                    df[col] = df[col].astype(np.float32)
                    
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory usage after optimization is: {end_mem:.2f} MB')
    print(f'Decreased by {100 * (start_mem - end_mem) / start_mem:.1f}%')
    return df

# Competition metric

In [3]:
def comp_metric_lgb(y_pred, data):
    """
    Custom Evaluation Metric cho LightGBM
    Trả về: (tên_metric, giá_trị, is_higher_better)
    """
    y_true = data.get_label()
    w = data.get_weight()
    
    denom = np.sum(w * (y_true ** 2))
    # Tránh chia cho 0
    if denom == 0:
        denom = 1e-15
        
    ratio = np.sum(w * ((y_true - y_pred) ** 2)) / denom
    
    # Clip giá trị trong khoảng [0, 1]
    clipped = np.clip(ratio, 0.0, 1.0)
    
    # Tính score cuối cùng
    val = 1.0 - clipped
    score = np.sqrt(val)
    
    # Điểm càng cao (gần 1) càng tốt
    return 'weighted_score', score, True

# Load and pre-processing

In [4]:
print("Loading data...")
df = pd.read_parquet('/kaggle/input/competitions/ts-forecasting/train.parquet') # Đổi đường dẫn tùy thực tế
df = reduce_mem_usage(df)

# Yêu cầu của bạn: Điền khuyết bằng 0
num_cols = [col for col in df.columns if col.startswith('feature_')]
print("Filling NaNs with 0...")
df[num_cols] = df[num_cols].fillna(0)

# Sắp xếp dữ liệu theo thời gian (BẮT BUỘC để tránh rò rỉ dữ liệu)
df = df.sort_values('ts_index').reset_index(drop=True)

# Encode các biến phân loại dạng chuỗi
categorical_cols = ['code', 'sub_code', 'sub_category', 'horizon']
print("Encoding categorical features...")
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# Định nghĩa các tập features
features = [col for col in df.columns if col.startswith('feature_')] + categorical_cols
target = 'y_target'
weight_col = 'weight'

Loading data...
Memory usage of dataframe is 3766.71 MB
Memory usage after optimization is: 1944.44 MB
Decreased by 48.4%
Filling NaNs with 0...
Encoding categorical features...


# Split

In [5]:
# 80% thời gian đầu làm Train, 20% thời gian cuối làm Validation
unique_times = df['ts_index'].unique()
split_idx = int(len(unique_times) * 0.8)
split_time = unique_times[split_idx]

print(f"Splitting data... Train <= {split_time}, Val > {split_time}")
train_df = df[df['ts_index'] <= split_time]
val_df = df[df['ts_index'] > split_time]

X_train, y_train, w_train = train_df[features], train_df[target], train_df[weight_col]
X_val, y_val, w_val = val_df[features], val_df[target], val_df[weight_col]

# Khởi tạo LightGBM Dataset (Gắn trực tiếp trọng số w_train, w_val)
train_data = lgb.Dataset(X_train, label=y_train, weight=w_train)
val_data = lgb.Dataset(X_val, label=y_val, weight=w_val, reference=train_data)

Splitting data... Train <= 2881, Val > 2881


# Train

In [6]:
params = {
    'objective': 'regression', # Bài toán dự báo giá trị liên tục
    'metric': 'None',          # Tắt metric mặc định để dùng Custom Metric
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 30,
    'max_depth': 8,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'random_state': 42,
    'verbose': -1
}

print("Start training...")
model = lgb.train(
    params,
    train_data,
    num_boost_round=2000,
    valid_sets=[train_data, val_data],
    feval=comp_metric_lgb,                  # Đưa Custom Metric vào đây
    callbacks=[
        lgb.early_stopping(stopping_rounds=100), # Dừng sớm nếu Validation score không tăng
        lgb.log_evaluation(period=50)            # In kết quả mỗi 50 vòng
    ]
)

Start training...
Training until validation scores don't improve for 100 rounds
[50]	training's weighted_score: 0.210701	valid_1's weighted_score: 0.117873
[100]	training's weighted_score: 0	valid_1's weighted_score: 0.131659
[150]	training's weighted_score: 0	valid_1's weighted_score: 0.134199
[200]	training's weighted_score: 0	valid_1's weighted_score: 0
[250]	training's weighted_score: 0	valid_1's weighted_score: 0
Early stopping, best iteration is:
[157]	training's weighted_score: 0	valid_1's weighted_score: 0.135297


# Submission

In [7]:
print("Đang tải dữ liệu Test...")
# Nhớ đổi đường dẫn đúng với file test của bạn
test_df = pd.read_parquet('/kaggle/input/competitions/ts-forecasting/test.parquet')

# 1. Tối ưu RAM (Dùng lại hàm reduce_mem_usage ở phần trước)
# Nhớ bỏ qua cột 'id' để không làm hỏng định dạng chuỗi của nó
test_df = reduce_mem_usage(test_df)

# 2. Xử lý Missing Values (Chỉ điền 0 cho các cột feature)
print("Đang fillna(0) cho các features...")
feature_cols = [col for col in test_df.columns if col.startswith('feature_')]
test_df[feature_cols] = test_df[feature_cols].fillna(0)

# 3. Mã hóa Categorical an toàn (Chống lỗi Unseen Labels)
print("Đang encode categorical features...")
categorical_cols = ['code', 'sub_code', 'sub_category', 'horizon']

for col in categorical_cols:
    le = label_encoders[col] # Lấy lại bộ biến đổi từ lúc Train
    
    # Tạo từ điển map từ tên chuỗi sang số
    mapping_dict = dict(zip(le.classes_, le.transform(le.classes_)))
    
    # Map dữ liệu test, nếu gặp giá trị mới lạ thì điền -1
    test_df[col] = test_df[col].astype(str).map(mapping_dict).fillna(-1).astype(int)

# 4. Gom các features lại theo đúng thứ tự lúc Train
features = feature_cols + categorical_cols
X_test = test_df[features]

# 5. DỰ ĐOÁN
print("Đang chạy dự đoán bằng mô hình LightGBM...")
# Dùng thuộc tính best_iteration để lấy vòng lặp tốt nhất (tránh overfit)
test_predictions = model.predict(X_test, num_iteration=model.best_iteration)

# 6. TẠO FILE SUBMISSION
print("Đang tạo file submission.csv...")
submission = pd.DataFrame({
    'id': test_df['id'],
    'prediction': test_predictions
})

# Lưu ra file CSV (không lưu index)
submission.to_csv('submission.csv', index=False)

print("HOÀN TẤT! Bạn đã có thể nộp file submission.csv lên Kaggle.")

Đang tải dữ liệu Test...
Memory usage of dataframe is 999.17 MB
Memory usage after optimization is: 516.15 MB
Decreased by 48.3%
Đang fillna(0) cho các features...
Đang encode categorical features...
Đang chạy dự đoán bằng mô hình LightGBM...
Đang tạo file submission.csv...
HOÀN TẤT! Bạn đã có thể nộp file submission.csv lên Kaggle.
